# imports

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from neuralforecast import NeuralForecast
import optuna
from sklearn.model_selection import TimeSeriesSplit
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import lightgbm as lgb

torch.set_float32_matmul_precision("medium")

from sklearn.metrics import root_mean_squared_error
from neuralforecast.losses.pytorch import MSE
from neuralforecast.models import PatchTST

# ============================================================
# GLOBAL SETTINGS FOR LIGHTWEIGHT HPO
# ============================================================
MAX_STEPS = 500
VAL_CHECK_STEPS = MAX_STEPS // 10


def split_train_val_test_global(df_nf, test_start, forecast_horizon, training_size, val_days=3, freq_minutes=15):
    """
    Split a long NeuralForecast dataframe into train / validation / test per series.

    Validation starts `val_days` before test_start and ends right before test_start.
    Test spans `forecast_horizon` steps starting at test_start.
    Training is the last `training_size` rows before validation starts.
    """
    test_start = pd.Timestamp(test_start)
    val_start = test_start - pd.Timedelta(days=val_days)
    test_end = test_start + pd.Timedelta(minutes=freq_minutes * forecast_horizon)

    train_parts = []
    val_parts = []
    test_parts = []

    expected_val_len = val_days * (24 * 60 // freq_minutes)

    for uid, g in df_nf.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)

        train_candidates = g[g["ds"] < val_start].copy()
        train_df_uid = train_candidates.iloc[-training_size:].copy()

        val_df_uid = g[(g["ds"] >= val_start) & (g["ds"] < test_start)].copy()
        test_df_uid = g[(g["ds"] >= test_start) & (g["ds"] < test_end)].copy()

        if len(train_df_uid) != training_size:
            raise ValueError(f"{uid}: expected {training_size} training rows, got {len(train_df_uid)}")

        if len(val_df_uid) != expected_val_len:
            raise ValueError(f"{uid}: expected {expected_val_len} validation rows, got {len(val_df_uid)}")

        if len(test_df_uid) != forecast_horizon:
            raise ValueError(f"{uid}: expected {forecast_horizon} test rows, got {len(test_df_uid)}")

        train_parts.append(train_df_uid)
        val_parts.append(val_df_uid)
        test_parts.append(test_df_uid)

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


def build_global_nf_df(df_all, home_cols):
    """
    Convert a wide household load dataframe into NeuralForecast long format.

    PatchTST here is used WITHOUT exogenous variables, so output is only:
    unique_id, ds, y
    """
    if not isinstance(df_all.index, pd.DatetimeIndex):
        raise ValueError("df_all index must be a DatetimeIndex.")

    df_base = df_all.reset_index().rename(columns={"timestamp": "ds"})

    df_nf = df_base.melt(
        id_vars=["ds"],
        value_vars=home_cols,
        var_name="unique_id",
        value_name="y"
    )

    df_nf = df_nf.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    return df_nf


def build_daily_profile_matrix(df_all, home_cols):
    df_tmp = df_all.copy()
    df_tmp["slot"] = df_tmp.index.hour * 4 + df_tmp.index.minute // 15

    profiles = []
    for home in home_cols:
        avg_profile = df_tmp.groupby("slot")[home].mean()
        avg_profile.name = home
        profiles.append(avg_profile)

    profile_df = pd.concat(profiles, axis=1).T
    profile_df.index.name = "home"

    return profile_df


def rolling_forecasting_validation_predictions(train_df, val_df, h, model_params, freq="15min"):
    rolling_train_df = train_df.copy()
    val_predictions = []

    val_starts = sorted(val_df["ds"].unique())[::h]

    for window_start in val_starts:

        model = PatchTST(
            h=h,
            input_size=model_params["input_size"],
            patch_len=model_params["patch_len"],
            stride=model_params["stride"],
            hidden_size=model_params["hidden_size"],
            n_heads=model_params["n_heads"],
            batch_size=model_params["batch_size"],
            learning_rate=model_params["learning_rate"],
            max_steps=MAX_STEPS,
            val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
            random_seed=42,
            loss=MSE(),
        )

        nf = NeuralForecast(models=[model], freq=freq)
        nf.fit(df=rolling_train_df)

        preds = nf.predict()
        val_predictions.append(preds)

        next_val_chunk = val_df[
            (val_df["ds"] >= window_start) &
            (val_df["ds"] < window_start + pd.Timedelta(minutes=15 * h))
        ].copy()

        rolling_train_df = pd.concat([rolling_train_df, next_val_chunk], ignore_index=True)
        rolling_train_df = rolling_train_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return pd.concat(val_predictions, ignore_index=True)


def compute_average_rmse_per_cluster(val_df, val_preds_df, pred_col="PatchTST"):
    val_compare_df = val_df.merge(val_preds_df, on=["unique_id", "ds"], how="left")

    rmse_rows = []
    for uid, g in val_compare_df.groupby("unique_id"):
        rmse_rows.append({
            "unique_id": uid,
            "RMSE": root_mean_squared_error(g["y"], g[pred_col])
        })

    rmse_per_home = pd.DataFrame(rmse_rows).sort_values("RMSE").reset_index(drop=True)
    avg_rmse_cluster = rmse_per_home["RMSE"].mean()

    return avg_rmse_cluster, rmse_per_home, val_compare_df


def objective(trial):
    model_params = {
        "input_size": trial.suggest_categorical("input_size", [
            forecast_horizon * 2,
            forecast_horizon * 3,
            forecast_horizon * 4,
        ]),
        "patch_len": trial.suggest_categorical("patch_len", [8, 12, 16, 24]),
        "stride": trial.suggest_categorical("stride", [4, 8, 12]),
        "hidden_size": trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "learning_rate": trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True),
    }

    try:
        val_preds_df = rolling_forecasting_validation_predictions(
            train_df=train_df,
            val_df=val_df,
            h=forecast_horizon,
            model_params=model_params,
            freq="15min"
        )

        avg_rmse_cluster, _, _ = compute_average_rmse_per_cluster(
            val_df=val_df,
            val_preds_df=val_preds_df,
            pred_col="PatchTST"
        )

        return avg_rmse_cluster

    except Exception as e:
        print(f"Trial failed: {e}")
        return float("inf")

# start

In [ ]:
import time

start_time = time.time()



project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

#countries = ["Germany", "Ireland", "Portugal"]
#countries = ["Germany"]
countries = ["Germany", "Ireland", "Portugal","Denmark"]

#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]



weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
#    "price_eur_kwh"
]


forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
feature_selection = True
plot_forecast = True
hyperparameter_opt = True
opt_trials = 20

# -------------------------
# Read JSON with forecast days
# -------------------------
with open(days_json_path, "r") as f:
    dataset_days = json.load(f)

# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    country_start_time = time.time()

    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    if country == "Denmark" and "price_eur_kwh" not in weather_cols:

        weather_cols.append("price_eur_kwh")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp")
    df_all = df_all.sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)

    # IMPORTANT:
    # PatchTST here does NOT use weather cols in df_nf
    df_nf = build_global_nf_df(df_all, home_cols)

    profile_df = build_daily_profile_matrix(df_all, home_cols)

    print(profile_df.head())
    print(profile_df.shape)

    scaler = StandardScaler()
    X_profile = scaler.fit_transform(profile_df)

    results = []
    for k in range(2, 6):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X_profile)
        score = silhouette_score(X_profile, labels)

        results.append({"k": k, "silhouette_score": score})

    results_df = pd.DataFrame(results).sort_values("silhouette_score", ascending=False)
    print(results_df)

    best_k = int(results_df.iloc[0]["k"])
    best_score = results_df.iloc[0]["silhouette_score"]

    print(f"Best k: {best_k}")
    print(f"Best silhouette score: {best_score:.4f}")

    best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    best_labels = best_kmeans.fit_predict(X_profile)

    cluster_profile_df = profile_df.copy()
    cluster_profile_df["cluster"] = best_labels

    print(cluster_profile_df["cluster"].sort_values())

    # -------------------------
    # Loop over days
    # -------------------------
    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")

        clusters = best_k
        cluster_test_preds_list = []

        for cluster in range(clusters):
            print(cluster)

            selected_cluster = cluster
            cluster_homes = cluster_profile_df.index[cluster_profile_df["cluster"] == selected_cluster].tolist()
            cluster_n_series = len(cluster_homes)

            print(f"cluster_n_series = {cluster_n_series}")
            print(f"Selected cluster: {selected_cluster}")
            print(f"Number of homes in cluster: {len(cluster_homes)}")
            print("Homes in cluster:")
            print(cluster_homes)

            # Keep weather in the original wide df only if you want,
            # but do NOT pass them to NeuralForecast
            df_cluster_wide = df_all[cluster_homes].copy()

            print("\nCluster-wide dataframe head:")
            print(df_cluster_wide.head())

            df_cluster_nf = build_global_nf_df(
                df_all=df_cluster_wide,
                home_cols=cluster_homes
            )

            print("\nCluster long-format dataset:")
            print(df_cluster_nf.head(10))

            print("\nColumns:")
            print(df_cluster_nf.columns.tolist())

            print("\nShape:")
            print(df_cluster_nf.shape)

            print("\nUnique homes in long dataset:")
            print(df_cluster_nf["unique_id"].unique())

            train_df, val_df, test_df = split_train_val_test_global(
                df_nf=df_cluster_nf,
                test_start=date,
                forecast_horizon=forecast_horizon,
                training_size=training_size,
                val_days=3,
                freq_minutes=15
            )

            for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
                start = df["ds"].min()
                end = df["ds"].max()
                print(f"{name}: {start} to {end} (Shape: {df.shape})")

            study = optuna.create_study(direction="minimize")
            study.optimize(objective, n_trials=opt_trials, show_progress_bar=True)

            print("Best avg RMSE:", study.best_value)
            print("Best params:", study.best_params)

            best_params = study.best_params

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            train_val_df = train_val_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

            final_patchtst = PatchTST(
                h=forecast_horizon,
                input_size=best_params["input_size"],
                patch_len=best_params["patch_len"],
                stride=best_params["stride"],
                hidden_size=best_params["hidden_size"],
                n_heads=best_params["n_heads"],
                batch_size=best_params["batch_size"],
                learning_rate=best_params["learning_rate"],
                max_steps=MAX_STEPS,
                val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
                random_seed=42,
                loss=MSE()
            )

            nf_final = NeuralForecast(
                models=[final_patchtst],
                freq="15min",
            )

            nf_final.fit(df=train_val_df)

            test_preds_df = nf_final.predict()

            test_preds_wide = test_preds_df.pivot(
                index="ds",
                columns="unique_id",
                values="PatchTST"
            ).sort_index()

            cluster_test_preds_list.append(test_preds_wide)

        final_test_preds_wide = pd.concat(cluster_test_preds_list, axis=1).sort_index()


        # ========================================================
        # COUNTRY RUNTIME
        # ========================================================
        country_runtime = time.time() - country_start_time

        print(
            f"\nTotal runtime for {country}: "
            f"{country_runtime:.2f} seconds"
        )

        # ========================================================
        # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
        # ========================================================
        json_path = pathlib.Path(project_path) / "Outputs" / f"time_spend_{country}.json"

        if json_path.exists():
            with open(json_path, "r") as f:
                runtime_dict = json.load(f)
        else:
            runtime_dict = {}

        if "Global" not in runtime_dict:
            runtime_dict["Global"] = {}

        runtime_dict["Global"]["PatchTST"] = country_runtime

        with open(json_path, "w") as f:
            json.dump(runtime_dict, f, indent=4)

        print(f"Saved/updated runtime JSON: {json_path}")



        # --------------------------------------------------
        # save final combined test predictions
        # --------------------------------------------------
        save_dir = pathlib.Path(project_path) / "Outputs" / "Global models" / "PatchTST"
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"prediction_PatchTST_{day_name}_{country}.csv"

        final_test_preds_wide.to_csv(save_path, index=True)

        print(f"Saved final_test_preds_wide to: {save_path}")



end_time = time.time()

total_seconds = end_time - start_time


In [ ]:
print(f"Time taken: {total_seconds:.4f} seconds")

file_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\time_spend.json"

# 1. Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# 2. Add TCN inside "Global"
data["Global"]["PatchTST"] = total_seconds

# 3. Save back (without disturbing structure)
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

# end 